# Experimental Journey · From Hough Circles to Pore Regions

**Empirical research log** supporting [`01_pore_segmentation_pipeline.ipynb`](01_pore_segmentation_pipeline.ipynb).

This notebook documents the **experiments, observations, decisions and remaining limitations** behind the final pipeline. It is intentionally not a second competing production pipeline.

**Evidence provenance:** experiments 1–16 are present in the original historical notebook; experiments 17–28 are reconstructed from recorded Jupyter runs and shared visual observations. Experiment 29 was proposed but **not executed**. Counts and porosity values are empirical outputs, **not ground-truth accuracy scores**.

## Research question and strategy

Initial circular Hough detection produced overlapping circles and missed smaller cavities. The goal evolved from **circle identification** to **connected pore-region segmentation**, including an explicit in-class suggestion to group circles belonging to the same pore.

The development strategy followed a sequence of controlled comparisons:
1. Improve detection recall and precision (multi-scale Hough, contrast, IoU-NMS).
2. Replace circular geometry with intensity-guided pore regions.
3. Correct specimen-boundary exclusions and internal holes.
4. Test generalization across ten slices.
5. Quantify the resulting 2D area fractions and inspect failure cases.

## Chronological experiment register

| Experiments | Hypothesis / approach | Observed result | Decision |
|---|---|---|---|
| **01–03** | Varied center spacing, peak thresholds and Hough score cutoffs. | Tradeoff: too strict misses pores; permissive settings generate duplicates. | **Exploratory** |
| **04–05** | Compared center/radius heuristic NMS and batch application to ten slices. | Reduced repeated circles, but did not reconstruct irregular pore shapes. | **Exploratory** |
| **06–07** | Tested permissive thresholds and additional candidate peaks. | Improved recall can introduce spurious detections. | **Exploratory** |
| **08–09** | Compared circle contrast and analytical circle-IoU NMS. | Retained contrast ≥20 and IoU threshold 0.30 in later pipeline. | **Retained** |
| **10** | Single Hough radius grid, contrast filtering and NMS. | 22–50 circles per slice; visibly undercounted small pores. | **Baseline** |
| **11** | Compared original radius grid with denser 4–88 px grid. | Recovered many smaller circles, with additional false-positive risk. | **Led to multi-scale** |
| **12** | Seeded local random-walker segmentation from Hough circles. | Irregular contours improved locally but too many pores missed. | **Rejected** |
| **13–14** | Local threshold, specimen mask, hole filling and varying morphological closing. | Fragmented/hollow pores; large closing merged neighboring cavities. | **Rejected** |
| **15–16** | Assigned Hough candidates to thresholded regions by center/overlap. | Missed and fragmented important pores; unsatisfactory. | **Rejected** |
| **17** | Independent 4–18 px and 20–88 px searches followed by shared NMS. | Image 01: 32 → 111 circles in an initial comparison; final tuned results differ. | **Retained** |
| **18–19** | Seeded grayscale flood around Hough circles, union of regions. | Image 09: 67 Hough circles → 44 connected regions. | **Retained** |
| **20** | Applied binary_fill_holes to reconstructed pore mask. | Removed artificial holes inside the central large pore, 44 regions unchanged. | **Retained** |
| **21–22** | Visualized masks with margins 0, 6, 12, 20; compared with unmasked Hough. | Image 09: 67 accepted and 6 border-rejected circles; peripheral pore rejected. | **Diagnostic** |
| **23–24** | Compared margin=12, margin=0, then convex hull. | Image 09: 44 → 47 → 48 regions; recovered lower peripheral pore. | **Retained** |
| **25** | Applied full Hough + convex mask + region growth + hole filling to all ten slices. | 35, 96, 92, 89, 75, 71, 65, 59, 50, 48 connected regions. | **Reference V1** |
| **26** | Compared opening radius 0–3 px on slices 02, 04 and 05. | No meaningful visual improvement; baseline preserved. | **Rejected** |
| **27** | Computed pore area inside specimen divided by specimen area. | 4.70%, 9.85%, 20.18%, 19.92%, 22.39%, 23.86%, 25.81%, 28.02%, 29.80%, 29.59%. | **Measured** |
| **28** | Visually inspected specimen and pore masks on slices 00, 02 and 09. | Specimen masks plausible; slice 02 pore mask includes a false bridge. | **Diagnostic** |
| **29** | Proposed ranking Hough seed-center intensities for slice 02. | Not executed/validated; deliberately left as future work. | **Proposed** |

## Reference outputs (10 slices)

These values were recorded from the reference pipeline, **not recomputed in this appendix**.

| Slice | Hough circles | Connected regions | Estimated 2D porosity |
|---:|---:|---:|---:|
| 00 | 48 | 35 | 4.70% |
| 01 | 119 | 96 | 9.85% |
| 02 | 121 | 92 | 20.18% |
| 03 | 103 | 89 | 19.92% |
| 04 | 89 | 75 | 22.39% |
| 05 | 81 | 71 | 23.86% |
| 06 | 73 | 65 | 25.81% |
| 07 | 64 | 59 | 28.02% |
| 08 | 64 | 50 | 29.80% |
| 09 | 73 | 48 | 29.59% |

The number of connected components depends on segmentation thresholds and may not equal the true number of physical pores. The early-slice fractions likely omit micro-pores; slice 02 may overestimate area due to region overgrowth.

## Key visual findings

**Multi-scale detection:** expanded small-pore coverage relative to the initial single-scale Hough search.

**Region reconstruction:** local flooding better followed non-circular boundaries than drawing circles. Union of region masks grouped overlapping detections, while hole filling removed false internal islands.

**Peripheral pore recovery:** the initial specimen mask developed a notch around a dark pore near the lower boundary. Using a convex hull restored it, but convexity is a geometric assumption that must be checked for other specimens.

**Generalization:** segmentation was visually strong on later slices (especially 03 and 09), but slice 02 developed an erroneous connected region through brighter material. Small pores were still missed in slices 00–01.

## Why other approaches were not retained

- **Adaptive local thresholding:** often generated hollow/fragmented regions, and aggressive closing merged separate pores.
- **Random walker:** improved selected boundaries but did not cover enough of the specimen.
- **Circle-to-label matching:** inherited the failures of the underlying threshold segmentation.
- **Morphological opening (1–3 px):** made little visible difference to the main failure cases.

This negative evidence is important: the selected pipeline was not chosen because every trial improved performance.

## Reproducibility and methodological cautions

- The main notebook contains the **runnable selected pipeline**, including the exact Jupyter install command.
- The original experimental notebook up to experiment 16 was a working research artifact, not a polished reproducible notebook. Keep it locally or archive separately if provenance is needed.
- The trial log here is a **faithful methodological synthesis**, not a claim that every historical run is fully reproducible from this appendix alone.
- TIFF files may be subject to data-sharing restrictions; publish only with permission.
- Do not label results as 3D porosity, classification accuracy or confirmed physical pore coalescence.

## Future experiments

**Priority 1 — validation:** manually annotate representative slices, including image 02, and report overlap metrics, false positives and missed pores.

**Priority 2 — robustness:** test the seed-intensity diagnostic proposed in experiment 29, then compare parameter sensitivity and alternative seed placement.

**Priority 3 — physics:** use acquisition metadata to verify scale and, if required, process the complete stacks for a true volumetric assessment.